In [27]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [29]:
!pip install scikit-learn
!pip install seaborn

  Using cached scikit_learn-1.9.1-cp314-cp314-win_amd64.whl.metadata (9.3 kB)
  Using cached scipy-1.18.1-cp314-cp314-win_amd64.whl.metadata (61 kB)
  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached threadpoolctl-3.7.0-py3-none-any.whl.metadata (24 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
Using cached scikit_learn-1.9.1-cp314-cp314-win_amd64.whl (8.4 MB)
Using cached joblib-1.6.0-py3-none-any.whl (306 kB)
Using cached cloudpickle-3.1.2-py3-none-any.whl (22 kB)
Using cached scipy-1.18.1-cp314-cp314-win_amd64.whl (37.4 MB)
Using cached threadpoolctl-3.7.0-py3-none-any.whl (26 kB)

   ------ --------------------------------- 1/6 [scipy]
   ------ --------------------------------- 1/6 [scipy]
   ------ --------------------------------- 1/6 [scipy]
   ------ --------------------------------- 1/6 [scipy]
   ------ --------------------------------- 1/6 [scipy]
   ------ --------------------------------- 1/6 [scipy]
   ------ ---------


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
df = pd.read_csv('train.csv')
df.head()

,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [6]:
df.shape

(42000, 785)

In [14]:
df['label'].max

<bound method Series.max of 0        1
1        0
2        1
3        4
4        0
        ..
41995    0
41996    1
41997    7
41998    6
41999    9
Name: label, Length: 42000, dtype: int64>

In [30]:
from sklearn.model_selection import train_test_split

y = df['label'].to_numpy()
data = df.drop(columns=['label']).to_numpy(dtype=np.float64) / 255.0

X_train, X_test, y_train, y_test = train_test_split(
    data,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 33600
Testing samples: 8400


In [20]:
epochs = 100
eta = 0.01

In [31]:
W1 = np.random.randn(784, 16) * 0.01
W2 = np.random.randn(16, 10) * 0.01

b1 = np.zeros(16)
b2 = np.zeros(10)

for epoch in range(epochs):
    lst = []

    for i in range(len(X_train)):
        # Forward propagation
        Z1 = np.dot(X_train[i], W1) + b1

        # Applying ReLU
        temp = np.maximum(0, Z1)

        Z2 = np.dot(temp, W2) + b2

        # Applying softmax
        exp_values = np.exp(Z2 - np.max(Z2))
        probabilities = exp_values / np.sum(exp_values)

        # Cross-entropy loss
        loss = -np.log(probabilities[y_train[i]] + 1e-15)
        lst.append(loss)

        # Backpropagation
        dZ2 = probabilities.copy()
        dZ2[y_train[i]] -= 1

        dW2 = np.outer(temp, dZ2)
        db2 = dZ2.copy()

        dA1 = W2 @ dZ2
        dZ1 = dA1 * (Z1 > 0)

        dW1 = np.outer(X_train[i], dZ1)
        db1 = dZ1.copy()

        # Update weights and biases
        W2 -= eta * dW2
        b2 -= eta * db2

        W1 -= eta * dW1
        b1 -= eta * db1

    cost = np.mean(lst)

    if epoch % 10 == 0:
        print(f"Epoch {epoch}: Loss = {cost:.6f}")

Epoch 0: Loss = 0.419261
Epoch 10: Loss = 0.139388
Epoch 20: Loss = 0.112412
Epoch 30: Loss = 0.093727
Epoch 40: Loss = 0.087890
Epoch 50: Loss = 0.078667
Epoch 60: Loss = 0.069609
Epoch 70: Loss = 0.061706
Epoch 80: Loss = 0.062225
Epoch 90: Loss = 0.064896


In [33]:
correct = 0

for i in range(len(X_train)):
    Z1 = np.dot(X_train[i], W1) + b1
    A1 = np.maximum(0, Z1)

    Z2 = np.dot(A1, W2) + b2

    prediction = np.argmax(Z2)

    if prediction == y_train[i]:
        correct += 1

train_accuracy = correct / len(X_train) * 100

print(f"Training Accuracy: {train_accuracy:.2f}%")
print(f"Test Accuracy: {accuracy:.2f}%")

Training Accuracy: 97.17%
Test Accuracy: 93.56%


In [32]:
correct = 0
test_losses = []

for i in range(len(X_test)):
    # Forward propagation
    Z1 = np.dot(X_test[i], W1) + b1
    A1 = np.maximum(0, Z1)

    Z2 = np.dot(A1, W2) + b2

    # Softmax
    exp_values = np.exp(Z2 - np.max(Z2))
    probabilities = exp_values / np.sum(exp_values)

    # Predicted digit
    prediction = np.argmax(probabilities)

    # Calculate loss
    loss = -np.log(probabilities[y_test[i]] + 1e-15)
    test_losses.append(loss)

    # Check prediction
    if prediction == y_test[i]:
        correct += 1

accuracy = (correct / len(X_test)) * 100
test_loss = np.mean(test_losses)

print(f"Test Accuracy: {accuracy:.2f}%")
print(f"Test Loss: {test_loss:.6f}")

Test Accuracy: 93.56%
Test Loss: 0.572148
